# BARAM 2026 — 모델링 v2: 격자 공간 feature + 감률, 축소 세트 (공식 지표 CV)

리서치(claudedocs/research_nwp_features_2026-07-09.md) 기반 v2:
- **더하기 (+10, 근거 ★★★~★★☆)**: GFS 9격자 공간 mean/std/gradient, LDAPS 16격자 동일, 감률 t850−t700·역전 t2m−t850.
- **빼기 (−31, CV 검증된 F3)**: 트리 불변 파생 3 + 죽은 NWP 19 + 풍속통계 중복 9.
- 결과 feature 수: 86 → **65** (요구사항: 변수 축소).

**검증**: expanding-window CV(g1·g2 2폴드, g3 1폴드), 공식 지표(`official_metric.py`).
비교: `A_full86` vs `B_lean55` vs `C_v2_65`. 승자로 최종 학습 + FICR 후처리 + `submission_v2.csv`.

## 0. 설정

In [1]:
import warnings; warnings.filterwarnings("ignore")
import json, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.ensemble import HistGradientBoostingRegressor as HGB
from sklearn.isotonic import IsotonicRegression
import wind_lib as W
from official_metric import group_scores, metric
plt.rcParams["figure.dpi"]=110; np.random.seed(42)

GROUPS=(1,2,3); FR={}; TGT={}; FR_TE={}
for g in GROUPS:
    df,tgt=W.load_train(g); TGT[g]=tgt
    FR[g]=W.add_spatial(W.build(df,g),"train")
    FR_TE[g]=W.add_spatial(W.build(W.load_test(g),g),"test")

BASE_ALL=[c for c in W.feature_cols(FR[1]) if c not in W.SPATIAL_COLS]+["pc_pred_cf"]
SETS={
 "A_full86": BASE_ALL,
 "B_lean55": W.lean_features(BASE_ALL),
 "C_v2_65":  W.lean_features(BASE_ALL)+W.SPATIAL_COLS,
}
for k,v in SETS.items(): print(k,len(v))

def lgbm(): return lgb.LGBMRegressor(objective="mae",n_estimators=600,learning_rate=0.03,num_leaves=63,
    min_child_samples=60,subsample=0.8,subsample_freq=1,colsample_bytree=0.7,reg_lambda=1.0,
    random_state=42,n_jobs=-1,verbose=-1)
def histgbm(): return HGB(loss="absolute_error",max_iter=600,learning_rate=0.03,max_leaf_nodes=63,
    l2_regularization=1.0,random_state=42)
def ens(tr,va,cols,tgt):
    lg=lgbm().fit(tr[cols],tr[tgt]); hg=histgbm().fit(tr[cols].to_numpy(),tr[tgt].to_numpy())
    return 0.5*(lg.predict(va[cols])+hg.predict(va[cols].to_numpy()))

A_full86 86
B_lean55 55
C_v2_65 65


## 1. Expanding-window CV (공식 지표)

In [2]:
FOLDS={1:[([2022],2023),([2022,2023],2024)],2:[([2022],2023),([2022,2023],2024)],3:[([2023],2024)]}
rows=[]
for sname,feats in SETS.items():
    for g in GROUPS:
        tgt=TGT[g]; cap=W.CAP[g]; fr=FR[g]; yr=fr.kst_dtm.dt.year
        for tys,vy in FOLDS[g]:
            tr=fr[yr.isin(tys)]; va=fr[yr==vy]
            iso=W.fit_powercurve(tr,tgt,cap); tr2=W.with_pc(tr,iso); va2=W.with_pc(va,iso)
            pred=np.clip(ens(tr2,va2,feats,tgt),0,cap)
            nm,fi=group_scores(va2[tgt].to_numpy(),pred,cap)
            rows.append(dict(set=sname,group=g,fold=vy,nmae=nm,ficr=fi))
    print("done",sname)
cv=pd.DataFrame(rows)

print("\n=== 폴드별 (nmae / ficr) ===")
print(cv.pivot_table(index=["group","fold"],columns="set",values=["nmae","ficr"]).round(4).to_string())

def total_2024(sname):
    sub=cv[(cv.set==sname)&(cv.fold==2024)]
    return 0.5*(1-sub.nmae.mean())+0.5*sub.ficr.mean()
def total_2023_g12(sname):
    sub=cv[(cv.set==sname)&(cv.fold==2023)]
    return 0.5*(1-sub.nmae.mean())+0.5*sub.ficr.mean()
print("\n=== 공식 총점 ===")
for s in SETS:
    print(f"  {s} ({len(SETS[s])}): 2024={total_2024(s):.4f}  2023(g12)={total_2023_g12(s):.4f}")

done A_full86


done B_lean55


done C_v2_65

=== 폴드별 (nmae / ficr) ===
               ficr                      nmae                 
set        A_full86 B_lean55 C_v2_65 A_full86 B_lean55 C_v2_65
group fold                                                    
1     2023   0.2515   0.2524  0.2493   0.1503   0.1518  0.1480
      2024   0.3448   0.3316  0.3516   0.1253   0.1270  0.1241
2     2023   0.3596   0.3535  0.3786   0.1394   0.1402  0.1352
      2024   0.4073   0.4032  0.4134   0.1263   0.1261  0.1254
3     2024   0.2549   0.2452  0.2447   0.1504   0.1535  0.1525

=== 공식 총점 ===
  A_full86 (86): 2024=0.6008  2023(g12)=0.5803
  B_lean55 (55): 2024=0.5956  2023(g12)=0.5785
  C_v2_65 (65): 2024=0.6013  2023(g12)=0.5862


## 2. 승자 판정 (2024 총점 + 2023 안정성)

In [3]:
tot={s:total_2024(s) for s in SETS}
winner=max(tot,key=tot.get)
# 동률(0.002) 시 더 작은 세트
TOL=0.002
cands=[s for s in SETS if tot[s]>=tot[winner]-TOL]
winner=min(cands,key=lambda s:len(SETS[s]))
print("승자:", winner, f"({len(SETS[winner])} feats, 2024 total={tot[winner]:.4f})")
print("근거: 총점", {s:round(tot[s],4) for s in SETS})
WIN_FEATS=SETS[winner]

# 공간 feature 중요도 확인 (C_v2 기준, group1)
g=1; tgt=TGT[g]; cap=W.CAP[g]; fr=FR[g]; yr=fr.kst_dtm.dt.year
tr=fr[yr.isin([2022,2023])]; va=fr[yr==2024]
iso=W.fit_powercurve(tr,tgt,cap); tr2=W.with_pc(tr,iso); va2=W.with_pc(va,iso)
m=lgbm().fit(tr2[SETS["C_v2_65"]],tr2[tgt])
imp=pd.Series(m.feature_importances_,index=SETS["C_v2_65"]).sort_values(ascending=False)
sp_rank={c:int((imp.index==c).argmax())+1 for c in W.SPATIAL_COLS}
print("\n공간·안정도 feature 중요도 순위 (65개 중):")
for c,r in sorted(sp_rank.items(),key=lambda x:x[1]): print(f"  #{r:2d} {c}")

승자: C_v2_65 (65 feats, 2024 total=0.6013)
근거: 총점 {'A_full86': np.float64(0.6008), 'B_lean55': np.float64(0.5956), 'C_v2_65': np.float64(0.6013)}



공간·안정도 feature 중요도 순위 (65개 중):
  # 4 gfs_inversion_2m_850
  # 5 gfs_lapse_850_700
  #10 gfs_ws100_grad_ew
  #12 gfs_ws100_grid_std
  #16 ldaps_ws10_grad_ns
  #18 gfs_ws100_grad_ns
  #20 ldaps_ws10_grid_std
  #24 gfs_ws100_grid_mean
  #26 ldaps_ws10_grid_mean
  #27 ldaps_ws10_grad_ew


## 3. FICR 후처리 (승자 feature로 재선택) & 최종 제출

`MODELING_FICR`과 동일 파이프라인: 학습기간 OOF로 debias/isotonic/nudge fit → 2024 총점 최대 조합 선택 → 전체 재학습 → `submission_v2.csv`.

In [4]:
def debias_fit(tr,tgt):
    d=tr[np.isfinite(tr["oof"])].copy(); d["resid"]=d[tgt]-d["oof"]
    d["lb"]=pd.cut(d["lead_h"],bins=[15,21,27,33,40],labels=False,include_lowest=True)
    d["wq"]=pd.qcut(d["gfs_wind_speed_100m_mean"],5,labels=False,duplicates="drop")
    return d.groupby(["lb","wq"])["resid"].mean(), d["resid"].mean()
def debias_apply(va,tbl,glob):
    v=va.copy()
    v["lb"]=pd.cut(v["lead_h"],bins=[15,21,27,33,40],labels=False,include_lowest=True)
    v["wq"]=pd.qcut(v["gfs_wind_speed_100m_mean"],5,labels=False,duplicates="drop")
    corr=np.array([tbl.get(k,glob) for k in zip(v["lb"],v["wq"])])
    return v["pred"].to_numpy()+corr
def iso_fit(tr,tgt):
    d=tr[np.isfinite(tr["oof"])]; ir=IsotonicRegression(out_of_bounds="clip",increasing=True)
    ir.fit(d["oof"].to_numpy(),d[tgt].to_numpy()); return ir
def nudge_fit(tr,tgt,cap):
    d=tr[np.isfinite(tr["oof"])]; yt=d[tgt].to_numpy(); yp=d["oof"].to_numpy()
    best=(1.0,0.0); bf=-1
    for s in np.linspace(0.90,1.15,26):
        for sh in np.linspace(-0.06,0.06,25)*cap:
            f=group_scores(yt,np.clip(yp*s+sh,0,cap),cap)[1]
            if f>bf: bf=f; best=(s,sh)
    return best

BASE={}
for g in GROUPS:
    tgt=TGT[g]; cap=W.CAP[g]; fr=FR[g]
    m_tr=fr.kst_dtm<W.VALID_START; m_va=fr.kst_dtm>=W.VALID_START
    iso=W.fit_powercurve(fr[m_tr],tgt,cap)
    tr=W.with_pc(fr[m_tr],iso); va=W.with_pc(fr[m_va],iso)
    val_pred=np.clip(ens(tr,va,WIN_FEATS,tgt),0,cap)
    oof=np.full(len(tr),np.nan); years=sorted(tr.kst_dtm.dt.year.unique())
    if len(years)>=2:
        for ty in years:
            m_in=tr.kst_dtm.dt.year!=ty; m_out=(tr.kst_dtm.dt.year==ty).to_numpy()
            iso2=W.fit_powercurve(tr[m_in],tgt,cap)
            a=W.with_pc(tr[m_in],iso2); b=W.with_pc(tr[tr.kst_dtm.dt.year==ty],iso2)
            oof[m_out]=np.clip(ens(a,b,WIN_FEATS,tgt),0,cap)
    else:
        n=len(tr); cut=int(n*0.7)
        oof[cut:]=np.clip(ens(tr.iloc[:cut],tr.iloc[cut:],WIN_FEATS,tgt),0,cap)
    BASE[g]=dict(tr=tr.assign(oof=oof), val=va.assign(pred=val_pred), cap=cap, tgt=tgt)

POST={}; STORE={}; choice={}
for g in GROUPS:
    d=BASE[g]; cap=d["cap"]; tgt=d["tgt"]; tr=d["tr"]; va=d["val"]; bp=va["pred"].to_numpy()
    tbl,glob=debias_fit(tr,tgt); ir=iso_fit(tr,tgt); s,sh=nudge_fit(tr,tgt,cap)
    p1=np.clip(debias_apply(va,tbl,glob),0,cap)
    cand={"P0_none":bp,"P1_debias":p1,"P2_isotonic":np.clip(ir.predict(bp),0,cap),
          "P3_nudge":np.clip(bp*s+sh,0,cap),"P4_deb_iso":np.clip(ir.predict(p1),0,cap),
          "P5_deb_nudge":np.clip(p1*s+sh,0,cap)}
    STORE[g]=dict(tbl=tbl,glob=glob,ir=ir,nudge=(s,sh))
    rows=[]
    for name,p in cand.items():
        nm,fi=group_scores(va[tgt].to_numpy(),p,cap); rows.append(dict(post=name,nmae=nm,ficr=fi,contrib=fi-nm))
    t=pd.DataFrame(rows).set_index("post"); POST[g]=t; choice[g]=t["contrib"].idxmax()
    print(f"group{g}: 선택={choice[g]}"); print(t.round(4).to_string())

def apply_post(g, frame, pred):
    cap=W.CAP[g]; st=STORE[g]; ch=choice[g]; fr2=frame.assign(pred=pred)
    if ch=="P0_none": return pred
    if ch=="P1_debias": return np.clip(debias_apply(fr2,st["tbl"],st["glob"]),0,cap)
    if ch=="P2_isotonic": return np.clip(st["ir"].predict(pred),0,cap)
    if ch=="P3_nudge": return np.clip(pred*st["nudge"][0]+st["nudge"][1],0,cap)
    q=np.clip(debias_apply(fr2,st["tbl"],st["glob"]),0,cap)
    if ch=="P4_deb_iso": return np.clip(st["ir"].predict(q),0,cap)
    return np.clip(q*st["nudge"][0]+st["nudge"][1],0,cap)

# 2024 공식 총점: 기준 vs 후처리
ans=pd.DataFrame({f"kpx_group_{g}":BASE[g]["val"][TGT[g]].to_numpy() for g in GROUPS})
p_base=pd.DataFrame({f"kpx_group_{g}":BASE[g]["val"]["pred"].to_numpy() for g in GROUPS})
p_post=pd.DataFrame({f"kpx_group_{g}":apply_post(g,BASE[g]["val"],BASE[g]["val"]["pred"].to_numpy()) for g in GROUPS})
t0=metric(ans,p_base); t1=metric(ans,p_post)
print(f"\n2024 공식 총점: 기준 {t0[0]:.4f} (1-NMAE {t0[1]:.4f}, FICR {t0[2]:.4f})")
print(f"               후처리 {t1[0]:.4f} (1-NMAE {t1[1]:.4f}, FICR {t1[2]:.4f})  Δ{t1[0]-t0[0]:+.4f}")

group1: 선택=P5_deb_nudge
                nmae    ficr  contrib
post                                 
P0_none       0.1241  0.3516   0.2275
P1_debias     0.1222  0.3524   0.2302
P2_isotonic   0.1247  0.3154   0.1907
P3_nudge      0.1227  0.4085   0.2858
P4_deb_iso    0.1234  0.3188   0.1953
P5_deb_nudge  0.1235  0.4128   0.2893
group2: 선택=P3_nudge
                nmae    ficr  contrib
post                                 
P0_none       0.1254  0.4134   0.2880
P1_debias     0.1244  0.4180   0.2936
P2_isotonic   0.1235  0.4068   0.2834
P3_nudge      0.1249  0.4370   0.3122
P4_deb_iso    0.1226  0.4097   0.2871
P5_deb_nudge  0.1253  0.4355   0.3102
group3: 선택=P5_deb_nudge
                nmae    ficr  contrib
post                                 
P0_none       0.1525  0.2447   0.0923
P1_debias     0.1461  0.2626   0.1165
P2_isotonic   0.1449  0.2740   0.1291
P3_nudge      0.1533  0.3072   0.1539
P4_deb_iso    0.1434  0.2976   0.1542
P5_deb_nudge  0.1703  0.3286   0.1584



2024 공식 총점: 기준 0.6013 (1-NMAE 0.8660, FICR 0.3366)
               후처리 0.6266 (1-NMAE 0.8605, FICR 0.3928)  Δ+0.0253


In [5]:
# 최종: 전체(2022-2024) 재학습 → test 예측 → 후처리 → 제출
out=W.load_test(1)[["forecast_id","kst_dtm"]].rename(columns={"kst_dtm":"forecast_kst_dtm"})
for g in GROUPS:
    tgt=TGT[g]; cap=W.CAP[g]
    iso=W.fit_powercurve(FR[g],tgt,cap)
    tr_all=W.with_pc(FR[g],iso); te=W.with_pc(FR_TE[g],iso)
    pred=np.clip(ens(tr_all,te,WIN_FEATS,tgt),0,cap)
    pred=apply_post(g, te, pred)
    out[f"kpx_group_{g}"]=pred
assert out.shape[0]==8760
for g in GROUPS:
    c=out[f"kpx_group_{g}"]; assert (c>=0).all() and (c<=W.CAP[g]).all()
out.to_csv("submission_v2.csv",index=False); print("saved submission_v2.csv",out.shape)

summary=dict(winner_feature_set=winner, n_features=len(WIN_FEATS),
    cv_total_2024={s:round(float(total_2024(s)),4) for s in SETS},
    chosen_post=choice,
    total_2024_base=round(float(t0[0]),4), total_2024_post=round(float(t1[0]),4),
    one_minus_nmae_post=round(float(t1[1]),4), ficr_post=round(float(t1[2]),4))
json.dump(summary,open("v2_summary.json","w"),ensure_ascii=False,indent=2)
print(json.dumps(summary,ensure_ascii=False,indent=2))

saved submission_v2.csv (8760, 5)
{
  "winner_feature_set": "C_v2_65",
  "n_features": 65,
  "cv_total_2024": {
    "A_full86": 0.6008,
    "B_lean55": 0.5956,
    "C_v2_65": 0.6013
  },
  "chosen_post": {
    "1": "P5_deb_nudge",
    "2": "P3_nudge",
    "3": "P5_deb_nudge"
  },
  "total_2024_base": 0.6013,
  "total_2024_post": 0.6266,
  "one_minus_nmae_post": 0.8605,
  "ficr_post": 0.3928
}


## 4. 결론

- 리서치 근거대로 **격자 공간 feature(★★★) + 감률(★★☆)** 을 추가하고 죽은 변수 31개를 제거한 v2를 CV로 판정.
- 승자 세트로 FICR 후처리까지 얹어 `submission_v2.csv` 생성.
- 공간 feature 중요도 순위(2절)로 실제 기여를 확인 — gradient가 상위권이면 문헌(-12.85%)의 방향이 우리 데이터에서도 재현되는 것.